# LoRA Practice: PyTorch (Manual) + Hugging Face PEFT

**Goal:** Train IMDb movie-review sentiment classification (Positive / Negative) using LoRA in two ways.

1. **Manual PyTorch:** create the LoRA A/B matrices, attach them to DistilBERT attention layers, and write the training loop.
2. **Hugging Face + PEFT:** apply LoRA with `LoraConfig` and train with `Trainer`.

The two sections use the same pretrained model family, dataset, tokenizer, and LoRA target layers so you can compare them.

**Beginner note:** This notebook uses a small subset first so it is practical to learn on a laptop or Colab. Increase the subset sizes later for better training.

## 0. Install the libraries

Run this cell once. If you are in Colab, restart the runtime if it asks you to.

In [11]:
!pip -q install -U transformers datasets accelerate peft evaluate

## 1. Imports and settings

In [12]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
)

MODEL_NAME = "distilbert-base-uncased"
MAX_LENGTH = 128
BATCH_SIZE = 8
EPOCHS = 5
LEARNING_RATE = 2e-4

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cuda


## 2. Load IMDb and make small practice splits

IMDb labels are:
- `0` = Negative
- `1` = Positive

We select a small number of examples to make the first run quicker. Later, try 5,000 or the full dataset.

In [13]:
dataset = load_dataset("stanfordnlp/imdb")

train_dataset = dataset["train"].shuffle(seed=42).select(range(2000))
test_dataset = dataset["test"].shuffle(seed=42).select(range(500))

print(train_dataset[0]["text"][:300])
print("Label:", train_dataset[0]["label"])

There is no relation at all between Fortier and Profiler but the fact that both are police series about violent crimes. Profiler looks crispy, Fortier looks classic. Profiler plots are quite simple. Fortier's plot are far more complicated... Fortier looks more like Prime Suspect, if we have to spot 
Label: 1


## 3. Load the tokenizer and tokenize the reviews

The tokenizer turns text into numbers. It does **not** train LoRA.

- `input_ids`: token IDs
- `attention_mask`: 1 for real tokens, 0 for padding

In [14]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_function(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=MAX_LENGTH,
    )

train_tokens = train_dataset.map(tokenize_function, batched=True)
test_tokens = test_dataset.map(tokenize_function, batched=True)

# Keep only the columns needed by the model.
train_tokens = train_tokens.remove_columns(["text"])
test_tokens = test_tokens.remove_columns(["text"])

# Dynamic padding: pad each batch only to its longest review.
collator = DataCollatorWithPadding(tokenizer=tokenizer)

train_loader = DataLoader(
    train_tokens,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collator,
)
test_loader = DataLoader(
    test_tokens,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collator,
)

print("Training batches:", len(train_loader))
print("Testing batches:", len(test_loader))

Training batches: 250
Testing batches: 63


# Part A — Manual LoRA with PyTorch

Here we write LoRA ourselves. The original DistilBERT weights are frozen. We train:
- LoRA matrix **A**
- LoRA matrix **B**
- the new IMDb classification head

For this practice notebook, LoRA is added to DistilBERT's Query and Value projection layers (`q_lin` and `v_lin`).

## 4. Write the LoRA layer

For an input vector `x`, the layer calculates:

\[
y = W_0x + \frac{\alpha}{r}B(Ax)
\]

- Original linear layer: `W0`
- `A`: reduces the hidden dimension to a small rank
- `B`: expands it back to the original dimension
- `r`: rank
- `alpha / r`: scaling factor

In [ ]:
class LoRALinear(nn.Module):
    def __init__(self, original_layer, rank=4, alpha=8, dropout=0.1):
        super().__init__()

        self.original_layer = original_layer
        self.rank = rank
        self.scale = alpha / rank
        self.dropout = nn.Dropout(dropout)

        input_size = original_layer.in_features
        output_size = original_layer.out_features

        # A: input_size -> rank
        self.lora_A = nn.Linear(input_size, rank, bias=False)

        # B: rank -> output_size
        self.lora_B = nn.Linear(rank, output_size, bias=False)

        # Common LoRA initialization: A random, B starts at zero.
        #Give the weights of A randomly generated starting values using Kaiming uniform initialization.
        #Because A needs starting weights to learn how to transform the input from 768 numbers into 4 numbers.
        #It is a parameter used internally by PyTorch's Kaiming uniform initialization to calculate the appropriate range of random values.
        #A is prepared with random starting values.B starts at zero, so LoRA initially contributes nothing.
        #Training teaches A and B how to adapt the pretrained model.
        nn.init.kaiming_uniform_(self.lora_A.weight, a=np.sqrt(5))
        nn.init.zeros_(self.lora_B.weight)

        # Keep the original layer frozen.
        for parameter in self.original_layer.parameters():
            parameter.requires_grad = False

    def forward(self, x):
        original_output = self.original_layer(x)#768 original dim
        lora_output = self.lora_B(self.lora_A(self.dropout(x)))#4 to 768 then do dropout from it
        return original_output + self.scale * lora_output#distilbert output +alpha/rank+loraoutput

## 5. Load DistilBERT and insert our LoRA layers

We replace every attention `q_lin` and `v_lin` linear layer with our `LoRALinear` wrapper.

In [ ]:
manual_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
)

# Freeze every parameter first.
for parameter in manual_model.parameters():
    parameter.requires_grad = False

# Add LoRA to Query and Value projections in every Transformer block
# After this We Get (Original Query layer + LoRA A/B)
#Now every Transformer block has LoRA added to:- Query - Value
for layer in manual_model.distilbert.transformer.layer:
    layer.attention.q_lin = LoRALinear(
        layer.attention.q_lin, rank=4, alpha=8
    )
    layer.attention.v_lin = LoRALinear(
        layer.attention.v_lin, rank=4, alpha=8
    )

# Train the task-specific classification head too.Unfreez classification head
#This accesses all the weights and biases inside the pre-classifier layer.
#We freeze the original pretrained DistilBERT weights, but keep LoRA A and B trainable. 
#We also unfreeze pre_classifier and classifier because they need to learn 
#how to convert the model's representations into our new task's predictions, 
#such as Positive or Negative IMDb reviews.
for parameter in manual_model.pre_classifier.parameters():
    parameter.requires_grad = True

for parameter in manual_model.classifier.parameters():
    parameter.requires_grad = True

manual_model.to(device)
#This counts all parameters that can be updated during training.so how much from the 
#all parameters we updated
trainable = sum(p.numel() for p in manual_model.parameters() if p.requires_grad)
total = sum(p.numel() for p in manual_model.parameters())
print(f"Trainable parameters: {trainable:,}")
print(f"Total parameters:     {total:,}")
print(f"Trainable percentage: {100 * trainable / total:.2f}%")

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Trainable parameters: 665,858
Total parameters:     67,028,738
Trainable percentage: 0.99%


## 6. Train the manual PyTorch model

Each batch follows this process:

1. Send reviews through DistilBERT + LoRA.
2. Compare predicted scores with the correct labels.
3. Calculate loss.
4. Backpropagate.
5. Update only trainable parameters.

In [ ]:
#IT WILL UPDATE ONLU THE PARAMETERS WHERE REQUIRES_GRAD=TRUE MEANS THE CLASSIFIER
optimizer = torch.optim.AdamW(
    [p for p in manual_model.parameters() if p.requires_grad],
    lr=LEARNING_RATE,
)

loss_function = nn.CrossEntropyLoss()

for epoch in range(EPOCHS):
    manual_model.train()
    total_loss = 0

    for batch in train_loader:
        batch = {key: value.to(device) for key, value in batch.items()}#move to device(GPU) this do only
        labels = batch.pop("labels")#remove the labels from the input dictionary and stores them separately.
        #so train didnt see the real label because it compare them with actual label so actual label should be there
        #in the batch

        optimizer.zero_grad()

        optimizer.zero_grad()

        outputs = manual_model(**batch)#It unpacks the dictionary and passes its contents as keyword arguments.
        #like outputss=input_ids=batch["input_ids"],attention_mask=batch["attention_mask"])
        loss = loss_function(outputs.logits, labels)#logits are real output and labels contain correct answer

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch + 1}: training loss = {total_loss / len(train_loader):.4f}")

Epoch 1: training loss = 0.5086
Epoch 2: training loss = 0.3680
Epoch 3: training loss = 0.3374
Epoch 4: training loss = 0.2922
Epoch 5: training loss = 0.2507


## 7. Evaluate the manual PyTorch model

In [18]:
manual_model.eval()
correct = 0
total = 0

with torch.no_grad():
    for batch in test_loader:
        batch = {key: value.to(device) for key, value in batch.items()}
        labels = batch.pop("labels")

        outputs = manual_model(**batch)
        predictions = outputs.logits.argmax(dim=1)

        correct += (predictions == labels).sum().item()
        total += labels.size(0)

manual_accuracy = correct / total
print(f"Manual PyTorch LoRA accuracy: {manual_accuracy:.3f}")

Manual PyTorch LoRA accuracy: 0.816


## 8. Try your own review with the manual model

In [19]:
def predict_manual(review):
    manual_model.eval()
    inputs = tokenizer(
        review,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_LENGTH,
    ).to(device)

    with torch.no_grad():
        logits = manual_model(**inputs).logits
        predicted_label = logits.argmax(dim=1).item()

    return "Positive" if predicted_label == 1 else "Negative"

print(predict_manual("This movie was wonderful. I really enjoyed it!"))
print(predict_manual("The story was boring and the acting was terrible."))

Positive
Negative


## 9. Save the manual LoRA weights

This saves the LoRA A/B matrices and the classification head, not a separate complete copy of DistilBERT. To use them later, load the same base model and rebuild the same LoRA structure before loading this file.

In [20]:
manual_lora_state = {}

for name, module in manual_model.named_modules():
    if isinstance(module, LoRALinear):
        manual_lora_state[name + ".lora_A.weight"] = module.lora_A.weight.detach().cpu()
        manual_lora_state[name + ".lora_B.weight"] = module.lora_B.weight.detach().cpu()

torch.save(
    {
        "lora": manual_lora_state,
        "pre_classifier": manual_model.pre_classifier.state_dict(),
        "classifier": manual_model.classifier.state_dict(),
    },
    "manual_lora_imdb.pt",
)

print("Saved manual_lora_imdb.pt")

Saved manual_lora_imdb.pt


---

# Part B — Hugging Face + PEFT

Now PEFT creates and inserts the LoRA layers for us. We do not write `LoRALinear` ourselves.

The steps are:
1. Load a fresh DistilBERT classification model.
2. Create `LoraConfig`.
3. Apply it with `get_peft_model`.
4. Train with Hugging Face `Trainer`.


## 10. Load a fresh model for PEFT

We create a **fresh** model so this experiment does not start with the weights changed in Part A.

In [21]:
from transformers import AutoModelForSequenceClassification

peft_base_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


## 11. Configure and apply LoRA

- `r=4`: LoRA rank.
- `lora_alpha=8`: scaling value.
- `lora_dropout=0.1`: dropout on the LoRA input.
- `target_modules`: the model layers that receive LoRA.

In [23]:
# Upgrade torchao to resolve incompatibility with peft
!pip install -U "torchao>0.16.0"

from peft import LoraConfig, get_peft_model, TaskType

lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=4,
    lora_alpha=8,
    lora_dropout=0.1,
    target_modules=["q_lin", "v_lin"],
)

peft_model = get_peft_model(peft_base_model, lora_config)
peft_model.print_trainable_parameters()
peft_model.to(device)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 41.1 MB/s  0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


W1109 10:15:10.037000 378 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


trainable params: 665,858 || all params: 67,620,868 || trainable%: 0.9847


PeftModelForSequenceClassification(
  (base_model): LoraModel(
    (model): DistilBertForSequenceClassification(
      (distilbert): DistilBertModel(
        (embeddings): Embeddings(
          (word_embeddings): Embedding(30522, 768, padding_idx=0)
          (position_embeddings): Embedding(512, 768)
          (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
          (dropout): Dropout(p=0.1, inplace=False)
        )
        (transformer): Transformer(
          (layer): ModuleList(
            (0-5): 6 x TransformerBlock(
              (attention): DistilBertSelfAttention(
                (q_lin): lora.Linear(
                  (base_layer): Linear(in_features=768, out_features=768, bias=True)
                  (lora_dropout): ModuleDict(
                    (default): Dropout(p=0.1, inplace=False)
                  )
                  (lora_A): ModuleDict(
                    (default): Linear(in_features=768, out_features=4, bias=False)
               

## 12. Prepare Hugging Face Trainer

`Trainer` manages the training and evaluation loop. `TrainingArguments` stores settings such as epochs, batch size, learning rate, and evaluation strategy.

In [ ]:
from transformers import TrainingArguments, Trainer
import inspect

# Handle the argument name used by different Transformers versions.
eval_argument = (
    "eval_strategy"
    if "eval_strategy" in inspect.signature(TrainingArguments).parameters
    else "evaluation_strategy"
)

training_settings = {
    "output_dir": "./peft_imdb_results",
    "learning_rate": LEARNING_RATE,
    "per_device_train_batch_size": BATCH_SIZE,
    "per_device_eval_batch_size": BATCH_SIZE,
    "num_train_epochs": EPOCHS,
    "weight_decay": 0.01,
    "logging_steps": 50,
    "save_strategy": "no",
    "report_to": "none",
}
training_settings[eval_argument] = "epoch"#This tells Trainer to evaluate the model after each epoch.
#This converts our dictionary into a TrainingArguments object.collected all our training settings into one object.training_args
training_args = TrainingArguments(**training_settings)

def calculate_accuracy(eval_prediction):
    logits, labels = eval_prediction
    predictions = np.argmax(logits, axis=-1)
    return {"accuracy": (predictions == labels).mean()}

trainer = Trainer(
    model=peft_model,
    args=training_args,
    train_dataset=train_tokens,
    eval_dataset=test_tokens,
    data_collator=collator,
    compute_metrics=calculate_accuracy,
)

## 13. Train with Trainer

In [25]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,0.386525,0.422470,0.818000
2,0.333671,0.455704,0.820000
3,0.342985,0.426577,0.832000
4,0.261980,0.450032,0.814000
5,0.335332,0.467748,0.816000


TrainOutput(global_step=1250, training_loss=0.3715735176086426, metrics={'train_runtime': 106.2427, 'train_samples_per_second': 94.124, 'train_steps_per_second': 11.766, 'total_flos': 336282286080000.0, 'train_loss': 0.3715735176086426, 'epoch': 5.0})

## 14. Evaluate the PEFT model

In [26]:
peft_results = trainer.evaluate()
print(peft_results)

Training Loss,Validation Loss,Epoch,Accuracy
0.335332,0.467748,5,0.816000


{'eval_loss': 0.46774810552597046, 'eval_accuracy': 0.816}


## 15. Predict your own review with PEFT

In [27]:
def predict_peft(review):
    peft_model.eval()
    inputs = tokenizer(
        review,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_LENGTH,
    ).to(device)

    with torch.no_grad():
        logits = peft_model(**inputs).logits
        predicted_label = logits.argmax(dim=1).item()

    return "Positive" if predicted_label == 1 else "Negative"

print(predict_peft("This movie was wonderful. I really enjoyed it!"))
print(predict_peft("The story was boring and the acting was terrible."))

Positive
Negative


## 16. Save and reload the PEFT adapter

PEFT provides `save_pretrained()` and `from_pretrained()` for saving and loading the adapter.

In [28]:
peft_model.save_pretrained("./imdb_lora_adapter")
tokenizer.save_pretrained("./imdb_lora_adapter")
print("Adapter saved.")

Adapter saved.


In [29]:
from peft import PeftModel

fresh_base = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
)

reloaded_model = PeftModel.from_pretrained(
    fresh_base,
    "./imdb_lora_adapter",
).to(device)

reloaded_model.eval()
print("Adapter loaded successfully.")

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Adapter loaded successfully.


---

# Part C — Compare and practise

## 17. What is different?

| Step | Manual PyTorch | Hugging Face + PEFT |
|---|---|---|
| Create A and B | You write the code | PEFT creates them |
| Choose target layers | You replace layers yourself | `target_modules` |
| Freeze base model | You control it | PEFT handles it |
| Training loop | You write it | `Trainer` handles it |
| Save adapter | You save selected weights | `save_pretrained()` |

**Important:** Both sections use PyTorch underneath. Hugging Face PEFT is a convenient library built to apply parameter-efficient fine-tuning methods to models.

## 18. Practice tasks

Try these one at a time:

1. Change `rank=4` / `r=4` to `r=8`.
2. Change `lora_alpha=8` to `16`.
3. Change `target_modules` to `["q_lin"]` and observe the trainable parameter count.
4. Increase the training subset from 2,000 to 5,000 examples.
5. Increase `EPOCHS` from 1 to 2.
6. Compare the two accuracy results. They may differ because training is stochastic.

## 19. Check your understanding

1. What does LoRA matrix A do?
2. What does LoRA matrix B do?
3. Why do we freeze the original pretrained weights?
4. What does `target_modules=["q_lin", "v_lin"]` mean?
5. What is the difference between `Trainer` and a manual training loop?
6. Does the IMDb dataset create DistilBERT's original vocabulary?

### Answers

1. A projects the hidden vector down to a smaller rank.
2. B projects the smaller vector back to the original output size.
3. To avoid updating all the large pretrained weights; we train a small number of parameters.
4. It tells PEFT which attention projection layers should receive LoRA.
5. With a manual loop you write the steps; `Trainer` manages much of that work.
6. No. The pretrained tokenizer converts IMDb text into token IDs. IMDb labels teach the model the sentiment task.

**Remember:** Manual PyTorch helps you understand how LoRA works. Hugging Face + PEFT helps you apply LoRA to real pretrained models with less code.